# Project

In [48]:
import pandas as pd
from thefuzz import process
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# For imputation
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

from scipy import stats
from sklearn.feature_selection import VarianceThreshold, RFE
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

import numpy as np
from sklearn.base import BaseEstimator, clone
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from typing import Dict

from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.neighbors import KNeighborsRegressor

from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor

import random
from copy import deepcopy
import time



In [49]:
train_df = pd.read_csv("https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/refs/heads/main/data/train.csv")

In [50]:
train_df.set_index("carID")

,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,,
69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.000000,0.0
53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.000000,0.0
6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.000000,0.0
29021,Ford,FIESTA,2018.0,12500,anual,9102.0,Petrol,145.0,65.700000,1.0,50.0,-2.340306,0.0
10062,BMW,2 Series,2019.0,22995,Manual,1000.0,Petrol,145.0,42.800000,1.5,97.0,3.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
37194,Mercedes,C Class,2015.0,13498,Manual,14480.0,etrol,125.0,53.300000,2.0,78.0,0.000000,0.0
6265,Audi,Q3,2013.0,12495,Semi-Auto,52134.0,Diesel,200.0,47.900000,2.0,38.0,2.000000,0.0
54886,Toyota,Aygo,2017.0,8399,Automatic,11304.0,Petrol,145.0,67.000000,1.0,57.0,3.000000,0.0


## Data cleaning

In [51]:
# TODO: Move the functions to a separate file

In [52]:
# String cleaning and Small numbers changes

def simple_processing(df):
    """
    Apply string cleaning, brand/model corrections, and fuzzy matching.
    These operations don"t require fitting on training data.
    """

    df = df.copy()
    # ============================================================================
    # SECTION 1: REFERENCE DATA SETUP
    # ============================================================================
    
    # Reference list of correct model names
    models = ["golf", "veloste", "caddy", "yaris", "q2", "fiesta", "2 series", "3 series", "a3", "octavia", 
              "passat", "focus", "insignia", "a class", "q3", "fabia", "ka+", "glc class", "i30", "c class", 
              "polo", "e class", "q5", "up", "c-hr", "mokka x", "corsa", "astra", "tt", "5 series", "aygo", 
              "4 series", "slk", "viva", "t-roc", "ecosport", "tucson", "x-class", "cl class", "ix20", "i20", 
              "rapid", "a1", "auris", "sharan", "adam", "x3", "a8", "gls class", "b-max", "a4", "kona", "i10", 
              "mokka", "s-max", "x2", "crossland x", "tiguan", "a5", "gle class", "zafira", "ioniq", "a6", 
              "mondeo", "yeti outdoor", "x1", "scala", "s class", "1 series", "kamiq", "kuga", "tourneo connect", 
              "q7", "gla class", "arteon", "sl class", "santa fe", "grandland x", "i800", "rav4", "touran", 
              "citigo", "roomster", "prius", "corolla", "b class", "kodiaq", "v class", "caddy maxi life", 
              "superb", "getz", "combo life", "beetle", "galaxy", "m3", "gtc", "x4", "ka", "ix35", 
              "grand tourneo connect", "m4", "tourneo custom", "z4", "x5", "meriva", "rs6", "verso", "touareg", 
              "shuttle", "cls class", "c-max", "puma", "cla class", "i40", "tiguan allspace", "6 series", 
              "caravelle", "karoq", "i3", "grand c-max", "t-cross", "a7", "golf sv", "agila", "gt86", "yeti", 
              "california", "land cruiser", "edge", "x6", "caddy life", "8 series", "fusion", "gl class", 
              "scirocco", "z3", "proace verso", "hilux", "amarok", "cc", "7 series", "avensis", "eos", "m class", 
              "grandland", "zafira tourer", "rs5", "r8", "mustang", "antara", "q8", "camry", "clk", "rs3", 
              "jetta", "kadjar", "sq5", "rs4", "supra", "i8", "x7", "sq7", "g class", "s3", "crossland", 
              "tigra", "escort", "glb class", "vivaro", "verso-s", "m5", "s4", "iq", "a2", "caddy maxi", 
              "streetka", "cascada", "accent", "s8", "rs", "golf s", "ranger", "vectra", "ampera", "fox", 
              "urban cruiser", "m2", "clc class", "m6", "s5", "terracan", "200", "220", "230", "NaN"]
    
    # Get unique short model names (2 characters) for separate handling
    short_models = [models[i] for i in range(len(models)) if len(models[i]) == 2]
    short_models = list(set(short_models))
    
    transmission_types = ["semi-auto", "manual", "automatic", "unkown", "NaN", "other"]
    fuel_types = ["petrol", "diesel", "hybrid", "electric", "other", "NaN"]
    
    # Brand name corrections mapping
    brand_mapping = {
        "vw": "vw",
        "v": "vw",
        "w": "vw",
        
        "toyota": "toyota",
        "toyot": "toyota",
        "oyota": "toyota",
        
        "audi": "audi",
        "aud": "audi",
        "udi": "audi",
        "ud": "audi",
        
        "ford": "ford",
        "for": "ford",
        "ord": "ford",
        "or": "ford",
        
        "bmw": "bmw",
        "bm": "bmw",
        "mw": "bmw",
        
        "skoda": "skoda",
        "skod": "skoda",
        "koda": "skoda",
        "kod": "skoda",
        
        "opel": "opel",
        "ope": "opel",
        "pel": "opel",
        "pe": "opel",
        
        "mercedes": "mercedes",
        "mercede": "mercedes",
        "ercedes": "mercedes",
        "ercede": "mercedes",
        
        "hyundai": "hyundai",
        "hyunda": "hyundai",
        "yundai": "hyundai",
        "yunda": "hyundai"
    }
    
    # ============================================================================
    # SECTION 2: INITIAL CLEANING (NO FITTING REQUIRED) -> no risk of data leakage
    # ============================================================================
        
    # Convert brand names to lowercase and removes all beginning and trailing whitespace (e.g. space) from the column
    df["Brand"] = df["Brand"].str.lower().str.strip()
    df["model"] = df["model"].str.lower().str.strip()
    df["transmission"] = df["transmission"].str.lower().str.strip()
    df["fuelType"] = df["fuelType"].str.lower().str.strip()
    
    # Replace NaN with string NaN to avoid errors in the fuzzy algorithm (cant match NaNs)
    df[["model", "transmission", "fuelType"]] = df[["model", "transmission", "fuelType"]].fillna("NaN")
    
    # 1.1 Fixing brands
    df["Brand"] = df["Brand"].map(brand_mapping)
    
    # 1.2 Fixing models
    # Similarity matching for Models, Transmission and fuel columns (Fuzzy Match)
    # Source for process.extractOne (fuzzy): https://github.com/seatgeek/thefuzz
    
    # VECTORIZED APPROACH: Only perform fuzzy matching once per unique value instead of per row
    
    # Models - handle different lengths separately
    unique_models = df["model"].unique()
    model_lookup = {}
    for val in unique_models:
        if pd.isna(val) or val == "NaN":
            model_lookup[val] = "NaN"
        elif len(val) > 2:  # Only perform fuzzy matching for models that have a name longer than 2 letters -> fuzzy will become fuzzy (unreliable) if names are to short
            model_lookup[val] = process.extractOne(val, models)[0]  # [0] because we get the name and score as a return -> score used for debugging
        elif len(val) == 2:  # Use the short names list for comparisons if the model names are 2 letters
            model_lookup[val] = process.extractOne(val, short_models)[0]
        else:  # We can define models with only one letter
            model_lookup[val] = "NaN"
    df["model"] = df["model"].map(model_lookup)
    
    # Transmission
    unique_trans = df["transmission"].unique()
    trans_lookup = {val: process.extractOne(val, transmission_types)[0] for val in unique_trans}
    df["transmission"] = df["transmission"].map(trans_lookup)
    
    # FuelType
    unique_fuel = df["fuelType"].unique()
    fuel_lookup = {val: process.extractOne(val, fuel_types)[0] for val in unique_fuel}
    df["fuelType"] = df["fuelType"].map(fuel_lookup)
    
    # Convert the str NaN values back to pd.NA for easier further processing and readability
    df["model"] = df["model"].replace("NaN", pd.NA)
    df["transmission"] = df["transmission"].replace(["unkown", "NaN", "other"], pd.NA)
    df["fuelType"] = df["fuelType"].replace(["other", "NaN"], pd.NA)
    
    #df["Brand"] = df["Brand"].fillna(df["Brand_mode"])  # rename new column
    #df.drop("Brand_mode", axis=1, inplace=True)  # remove the old brand column
    
    ################################################################################
    # Simple Number Cleaning
    ################################################################################

    # Cleaning numeric columns
    df["year"] = df["year"].round(0)
    
    # Create the new column
    df["stated_no_damage"] = ~df["hasDamage"].astype(bool)
    df = df.drop(["hasDamage"], axis=1)

    # Round year to integer (no fractional years)
    df["year"] = df["year"].round()
    
    # Mileage: take absolute value and round
    # Some imputation might produce small negative values
    df["mileage"] = abs(df["mileage"].round())
    
    # Tax: take absolute value and round
    df["tax"] = abs(df["tax"].round())
    
    # MPG: round to 1 decimal place 
    df["mpg"] = abs(df["mpg"].round(1))
    
    # Engine size: round to 1 decimal place
    df["engineSize"] = abs(df["engineSize"].round(1))
    
    # Paint quality correction (domain-specific business logic)
    # Assumption based on data exploration:
    # - Values < 4 likely had decimal point in wrong place (e.g., 3.5 -> 35%)
    # - Values > 100 likely have erroneous leading 1 (e.g., 185 -> 85%)
    def fix_paint_quality(x):
        if x < 4:
            return x * 10
        elif x > 100:
            return x - 100
        else:
            return x
    
    df["paintQuality%"] = df["paintQuality%"].apply(fix_paint_quality).round()
    
    # Previous owners: take absolute value and round to integer
    df["previousOwners"] = abs(df["previousOwners"].round())
    
    return df

In [53]:
def mode_imputation(df):
    """
    Get the most frequent brand for each model -> returns df with model and brand
    """
    brand_models = df.groupby("model_transformed")["Brand_transformed"].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else pd.NA)
    df = pd.merge(df, brand_models, on="model_transformed", how="left", suffixes=("", "_mode"))  # add the model and brand df to our main df (onyl add the brand columns, join on model)
    
    df["Brand_transformed"] = df["Brand_transformed"].fillna(df["Brand_transformed_mode"])  # rename new column
    df.drop("Brand_transformed_mode", axis=1, inplace=True)  # remove the old brand column
    

    return df

In [54]:
# Categorical featue Encoding
def fit_transform_encoding(df):
    """
    Fit label encoders for categorical columns on training data.
    """
    
    encoders = {
        "Brand": LabelEncoder(),
        "model": LabelEncoder(),
        "transmission": LabelEncoder(),
        "fuelType": LabelEncoder()
    }
    
    # Fit each encoder on the corresponding column
    """encoders["brand"].fit(df["Brand"])
    encoders["model"].fit(df["model"])
    encoders["transmission"].fit(df["transmission"])
    encoders["fuelType"].fit(df["fuelType"])"""

    columns = ["Brand", "model", "transmission", "fuelType"]

    # Code adapted from: https://stackoverflow.com/questions/36808434/label-encoder-encoding-missing-values

    for column in columns:
        # Get non-null string values
        mask = df[column].notna() & (df[column].apply(type) == str)
        
        # Fit encoder on unique non-null values
        fit_by = df.loc[mask, column].unique()
        encoders[column].fit(fit_by)
        
        # Transform only non-null values (vectorized)
        new_col_name = column + "_transformed"
        df[new_col_name] = pd.NA  # Initialize with NA
        df.loc[mask, new_col_name] = encoders[column].transform(df.loc[mask, column])
        
        # Convert to nullable integer
        df[new_col_name] = df[new_col_name].astype("Int64")

    df = df.drop(columns, axis=1)
    return df, encoders

In [55]:
# Train imputer on train

def fit_imputer(df, fast=True): 
    # Select estimator based on speed/accuracy tradeoff
    if fast:
        # Use default BayesianRidge (fast, ~1 second)
        estimator = None
    else:
        # Use Random Forest for better accuracy with complex relationships (~2 minutes)
        estimator = RandomForestRegressor(
            n_estimators=20,      # Limited trees for speed
            max_depth=10,         # Prevent overfitting
            random_state=12       # Reproducibility
        )

    # TODO: for later
    # Test if we perform better if we use numerical and categorical imputers separately
    
    # Initialize imputer
    imputer = IterativeImputer(
        estimator=estimator,
        max_iter=10,                    # Number of imputation rounds
        random_state=12,                # For reproducibility
        initial_strategy="mean"         # Initial fill before iterative process
    )
    
    # TODO: Test other categorical imputers like: missForest, datawig
    """imputer = IterativeImputer(
        estimator=RandomForestClassifier(),
        max_iter=10,                    # Number of imputation rounds
        random_state=12,                # For reproducibility
        initial_strategy="most_frequent"         # Initial fill before iterative process
        )"""

    # FIT on training data
    # CRITICAL: We fit on data that still has missing values!
    # The imputer learns patterns of missingness and relationships

    imputer.fit(df)
    
    return imputer

In [56]:
def apply_imputer(df, imputer):
    """
        Apply the pretrained imputer to the dataframe
    """

    imputed_values = imputer.transform(df)

    df[df.columns] = imputed_values

    # TODO: rounding is not the best approach as the imputers prediction are continues thus 1.2 doesnt mean the value is closer to 1 than 2
    # However, rounding is the quickest way to fix this for now
    df[["mpg", "engineSize"]] = abs(df[["mpg", "engineSize"]]).round(1)
    try: 
        df[["year", "price", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]] = abs(df[["year", "price", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]]).round(0).astype(int)
    except KeyError: # will raise keyError if we run it on testing data as it has no price column
        df[["year", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]] = abs(df[["year", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]]).round(0).astype(int)
    
    return df

In [57]:
def decode(df, encoders):
    # Iterative imputer produces ~20-30 values that are outside of the range of the encoder
    # The simplest fix is to clip does values back into the range of the encoder
 

    df["Brand_transformed"] = df["Brand_transformed"].clip(lower=0, upper=encoders["Brand"].classes_.shape[0]-1).astype(int)
    df["Brand"] = encoders["Brand"].inverse_transform(df["Brand_transformed"])

    df["transmission_transformed"] = df["transmission_transformed"].clip(lower=0, upper=encoders["transmission"].classes_.shape[0]-1).astype(int)
    df["transmission"] = encoders["transmission"].inverse_transform(df["transmission_transformed"])
        
    # Use clip with the information of the fitted encoder, .classes_.shape gives us the dimension of the labels the encoder uses [0] is the rows - 1 because we start clipping at 0
    df["model_transformed"] = df["model_transformed"].clip(lower=0, upper=encoders["model"].classes_.shape[0]-1).astype(int)
    df["model"] = encoders["model"].inverse_transform(df["model_transformed"])
    
    df["fuelType_transformed"] = df["fuelType_transformed"].clip(lower=0, upper=encoders["fuelType"].classes_.shape[0]-1).astype(int)
    df["fuelType"] = encoders["fuelType"].inverse_transform(df["fuelType_transformed"])
    

    df.drop(columns=["Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"], inplace=True)

    return df

### Workflow for Train and Validation Sets

#### Encoding

In [58]:
# Fix typos and small numeric cleanup
df = simple_processing(train_df)
# Encode cateogrical columns and replace the str with int columns. Return fitted encoders for decoding at the end

df_encoded, encoders = fit_transform_encoding(df)

#### Creating the stratification column

In [59]:
# Create Categorical price column with 0 < 1 < 2 for the price
df_encoded["price_cat"] = pd.qcut(df_encoded["price"], 3, labels=False)

# Combine the 10 unique brand values (1-9 & NA) with the 3 unique price category values (0-2)
stratify_col = df_encoded["Brand_transformed"].astype(str) + "_" + df_encoded["price_cat"].astype(str)

#### Imputation and decoding

In [60]:
# Train validation split
train_split, validation_split = train_test_split(df_encoded, test_size=0.2, random_state=42, stratify=stratify_col)
# train_split, validation_split = train_test_split(df_encoded, test_size=0.2, random_state=42)

train_split = mode_imputation(train_split)
validation_split = mode_imputation(validation_split)

# Train imputer on train (data leakage risk thus only train on train_split)
imputer = fit_imputer(train_split)

# Apply trained imputer to both datasplits
imputed_train = apply_imputer(train_split, imputer)
imputer_test = apply_imputer(validation_split, imputer)

# Decode encoded columns using the fitted encoders
train_processed = decode(imputed_train, encoders)
validation_processed = decode(imputer_test, encoders)

### Workflow for Seperated Testing Dataset

In [61]:
test_df = pd.read_csv("https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/refs/heads/main/data/test.csv")

In [62]:
test_df.set_index("carID")

,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,
89856,Hyundai,I30,2022.878006,Automatic,30700.000000,petrol,205.0,41.5,1.6,61.0,3.0,0.0
106581,VW,Tiguan,2017.000000,Semi-Auto,-48190.655673,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
80886,BMW,2 Series,2016.000000,Automatic,36792.000000,Petrol,125.0,51.4,1.5,94.0,2.0,0.0
100174,Opel,Grandland X,2019.000000,Manual,5533.000000,Petrol,145.0,44.1,1.2,77.0,1.0,0.0
81376,BMW,1 Series,2019.000000,Semi-Auto,9058.000000,Diesel,150.0,51.4,2.0,45.0,4.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
105775,VW,Tiguan,2017.000000,Manual,27575.000000,Petrol,145.0,46.3,1.4,94.0,1.0,0.0
81363,BMW,X2,2020.000000,Automatic,1980.000000,Petrol,145.0,34.0,2.0,39.0,3.0,0.0
76833,Audi,Q5,2019.000000,Semi-Auto,8297.000000,Diesel,145.0,38.2,2.0,88.0,4.0,0.0


In [63]:
test = simple_processing(test_df)
test_df_encoded, test_encoders = fit_transform_encoding(test)


df_encoded_no_price = df_encoded.drop(["price", "price_cat"], axis=1)
test_imputers = fit_imputer(df_encoded_no_price) # we use the full encoded training dataframe to train the imputers

# Apply trained imputer to both datasplits
test_df_imputed = apply_imputer(test_df_encoded, test_imputers)

test_processed = decode(test_df_imputed, test_encoders)


In [125]:
train_val_models = set(pd.concat([train_processed, validation_processed])["model"].unique())
test_models = set(test_processed["model"].unique())

missing_models = train_val_models - test_models


In [126]:
list(missing_models)


['streetka',
 '200',
 'urban cruiser',
 'getz',
 '230',
 '220',
 'a2',
 'kadjar',
 'escort',
 'caddy maxi',
 'accent',
 'ranger',
 'verso-s']

## Feature Engineering

In [64]:
train_processed = train_processed.set_index("carID")
validation_processed = validation_processed.set_index("carID")
train_processed.index = train_processed.index.astype(int)
validation_processed.index = validation_processed.index.astype(int)


In [65]:
X_train = train_processed.drop(columns=['price'])
y_train = train_processed['price']
X_val = validation_processed.drop(columns=['price'])
y_val = validation_processed['price']

In [66]:
def minimal_features(df):
    df = df.copy()
    df['age'] = 2024 - df['year']
    df = df.drop(columns=["year"])
    df['mileage_per_year'] = df['mileage'] / (df['age'] + 1)
    df['efficiency_ratio'] = df['mpg'] / (df['engineSize'] + 0.1)
    df['age_mileage'] = df['age'] * df['mileage'] / 100000
    df['condition_score'] = (df['paintQuality%']/100) * 0.5 + df['stated_no_damage'].astype(int) * 0.5

    return df

In [67]:
X_train = minimal_features(X_train)
X_val = minimal_features(X_val)

In [68]:
X_train = X_train.drop(columns=["price_cat"])
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60778 entries, 44735 to 59767
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mileage           60778 non-null  int32  
 1   tax               60778 non-null  int32  
 2   mpg               60778 non-null  float64
 3   engineSize        60778 non-null  float64
 4   paintQuality%     60778 non-null  int32  
 5   previousOwners    60778 non-null  int32  
 6   stated_no_damage  60778 non-null  float64
 7   Brand             60778 non-null  object 
 8   transmission      60778 non-null  object 
 9   model             60778 non-null  object 
 10  fuelType          60778 non-null  object 
 11  age               60778 non-null  int32  
 12  mileage_per_year  60778 non-null  float64
 13  efficiency_ratio  60778 non-null  float64
 14  age_mileage       60778 non-null  float64
 15  condition_score   60778 non-null  float64
dtypes: float64(7), int32(5), object(4)
memory

In [69]:
X_val = X_val.drop(columns=["price_cat"])
X_val.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15195 entries, 10742 to 43030
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mileage           15195 non-null  int32  
 1   tax               15195 non-null  int32  
 2   mpg               15195 non-null  float64
 3   engineSize        15195 non-null  float64
 4   paintQuality%     15195 non-null  int32  
 5   previousOwners    15195 non-null  int32  
 6   stated_no_damage  15195 non-null  float64
 7   Brand             15195 non-null  object 
 8   transmission      15195 non-null  object 
 9   model             15195 non-null  object 
 10  fuelType          15195 non-null  object 
 11  age               15195 non-null  int32  
 12  mileage_per_year  15195 non-null  float64
 13  efficiency_ratio  15195 non-null  float64
 14  age_mileage       15195 non-null  float64
 15  condition_score   15195 non-null  float64
dtypes: float64(7), int32(5), object(4)
memory

In [70]:
y_train

carID
44735    33569
44774    27699
36653    24400
22889     6391
62059    11386
         ...  
71271     9798
7862     16750
33501     7900
68727    29100
59767     9999
Name: price, Length: 60778, dtype: int32

In [71]:
y_val

carID
10742    23975
33830    12991
2417      3990
4544     27990
73794    14578
         ...  
8206     11850
71796    27490
72896     3999
23839    17495
43030    29062
Name: price, Length: 15195, dtype: int32

## Test of the models on the full dataset

In [72]:
class BrandModelTrainer:
    def __init__(self, estimator):
        self.estimator = estimator
        self.brand_models = {}
        self.feature_cols = None

    def fit(self, X_train, y_train):
        self.feature_cols = [c for c in X_train.columns if c != "Brand"]
        print(f"Training models for {len(X_train['Brand'].unique())} brands...\n")

        for brand in X_train["Brand"].unique():
            mask = X_train["Brand"] == brand
            Xb = X_train.loc[mask, self.feature_cols]
            yb = y_train[mask]

            numeric_cols = Xb.select_dtypes(include=["int64", "int32" ,"float64"]).columns.tolist()
            categorical_cols = Xb.select_dtypes(include=["object", "category"]).columns.tolist()

            preprocessor = ColumnTransformer([
                ("num", RobustScaler(), numeric_cols),
                ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
            ])

            model = Pipeline([
                ("preprocess", preprocessor),
                ("estimator", clone(self.estimator))
            ])

            model.fit(Xb, yb)
            self.brand_models[brand] = model
            print(f"  ✓ {brand} done.")

        return self

    def predict(self, X):
        preds = np.zeros(len(X))
        for brand, model in self.brand_models.items():
            mask = X["Brand"] == brand
            if mask.sum() == 0:
                continue
            Xb = X.loc[mask, self.feature_cols]
            preds[mask] = model.predict(Xb)
        return preds

    def evaluate_train(self, X_train, y_train):
        y_pred = self.predict(X_train)
        rmse = np.sqrt(mean_squared_error(y_train, y_pred))
        mae = mean_absolute_error(y_train, y_pred)
        r2 = r2_score(y_train, y_pred)
        print("\nTraining Set Performance (Overall):")
        print(f"  RMSE: {rmse:.2f}")
        print(f"  MAE:  {mae:.2f}")
        print(f"  R²:   {r2:.4f}")
        return {"RMSE": rmse, "MAE": mae, "R²": r2}

    def evaluate(self, X_val, y_val):
        y_pred = self.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = mean_absolute_error(y_val, y_pred)
        r2 = r2_score(y_val, y_pred)
        print("\nValidation Set Performance (Overall):")
        print(f"  RMSE: {rmse:.2f}")
        print(f"  MAE:  {mae:.2f}")
        print(f"  R²:   {r2:.4f}")
        return {"RMSE": rmse, "MAE": mae, "R²": r2}

    def evaluate_by_brand(self, X, y, split_name="Validation"):
        y_pred = self.predict(X)
        results = []
        for brand in X["Brand"].unique():
            mask = X["Brand"] == brand
            y_true_b = y[mask]
            y_pred_b = y_pred[mask]
            rmse = np.sqrt(mean_squared_error(y_true_b, y_pred_b))
            mae = mean_absolute_error(y_true_b, y_pred_b)
            r2 = r2_score(y_true_b, y_pred_b)
            results.append({"Brand": brand, "N": len(y_true_b), "RMSE": rmse, "MAE": mae, "R²": r2})
        df = pd.DataFrame(results).sort_values("RMSE")
        print(f"\n{split_name} Performance per Brand:")
        print(df.to_string(index=False))
        return df

    def evaluate_train_by_brand(self, X_train, y_train):
        return self.evaluate_by_brand(X_train, y_train, split_name="Training")
    
    def save_predictions(self, X, path):
        preds = self.predict(X)
        df = pd.DataFrame({
            "CarID": X["CarID"].values,
            "price": preds
        })
        df.to_csv(path, index=False)
        print(f"File salvato in: {path}")
        return df


In [73]:
raise SystemExit("Stop before training the models")

SystemExit: Stop before training the models

C:\Users\liber\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Linear Regression

In [74]:
LR = LinearRegression()
trainer = BrandModelTrainer(LR)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ mercedes done.
  ✓ ford done.
  ✓ opel done.
  ✓ skoda done.
  ✓ audi done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ bmw done.

Training Set Performance (Overall):
  RMSE: 3380.85
  MAE:  2014.80
  R²:   0.8789

Validation Set Performance (Overall):
  RMSE: 3488.30
  MAE:  2061.42
  R²:   0.8737

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7633 1553.487221 1075.327688 0.809318
  toyota  3770 1907.315967 1178.019716 0.905757
    ford 13120 2030.106966 1412.417309 0.821233
   skoda  3504 2071.581305 1475.058073 0.884629
 hyundai  2728 2125.775692 1494.385483 0.872634
      vw  8488 2851.656086 1992.541466 0.862518
    audi  5979 4164.132833 2710.427594 0.872121
     bmw  6032 4263.054881 2737.450213 0.864427
mercedes  9524 5601.268326 3401.902605 0.746194

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1912 1556.566420 1100.204792 0.811613
    ford 3272

,Brand,N,RMSE,MAE,R²
7,opel,1912,1556.566420,1100.204792,0.811613
4,ford,3272,2007.789638,1408.503335,0.818991
1,hyundai,676,2109.611642,1498.986787,0.874111
6,toyota,948,2326.030448,1257.583011,0.888588
3,vw,2118,2988.753071,2085.102039,0.856886
8,skoda,882,3058.561033,1542.063744,0.796293
0,bmw,1514,4064.865621,2714.628644,0.860412
2,audi,1489,4238.717697,2701.409287,0.870185
5,mercedes,2384,5792.752645,3564.143687,0.745079


### ElasticNet

In [75]:
Elastic = ElasticNet()
trainer = BrandModelTrainer(Elastic)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)        # Performance VALID

Training models for 9 brands...

  ✓ mercedes done.
  ✓ ford done.
  ✓ opel done.
  ✓ skoda done.
  ✓ audi done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ bmw done.

Training Set Performance (Overall):
  RMSE: 4877.07
  MAE:  3004.49
  R²:   0.7481

Validation Set Performance (Overall):
  RMSE: 4956.03
  MAE:  3039.99
  R²:   0.7450

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7633 2276.768085 1705.542112 0.590426
    ford 13120 2773.118187 1993.085741 0.666430
 hyundai  2728 3400.891063 2460.719099 0.674010
   skoda  3504 3416.761384 2451.956322 0.686149
  toyota  3770 3564.493160 2388.972686 0.670846
      vw  8488 4444.784957 3052.834859 0.665996
    audi  5979 6342.491208 3783.407764 0.703333
mercedes  9524 6744.941632 4307.569557 0.631969
     bmw  6032 7544.506573 4902.116314 0.575388

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1912 2236.901130 1694.842772 0.610948
    ford 3272

,Brand,N,RMSE,MAE,R²
7,opel,1912,2236.901130,1694.842772,0.610948
4,ford,3272,2667.850122,1969.886276,0.680414
1,hyundai,676,3355.629894,2387.951904,0.681484
8,skoda,882,4209.466075,2651.006135,0.614143
6,toyota,948,4372.783516,2564.539942,0.606253
3,vw,2118,4591.360191,3184.666880,0.662258
2,audi,1489,6287.644806,3761.883497,0.714352
0,bmw,1514,6980.504921,4669.488881,0.588349
5,mercedes,2384,7145.177273,4491.147713,0.612152


### KNR

In [77]:
KNR = KNeighborsRegressor()
trainer = BrandModelTrainer(KNR)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ mercedes done.
  ✓ ford done.
  ✓ opel done.
  ✓ skoda done.
  ✓ audi done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ bmw done.

Training Set Performance (Overall):
  RMSE: 2320.16
  MAE:  1345.63
  R²:   0.9430

Validation Set Performance (Overall):
  RMSE: 3004.83
  MAE:  1691.97
  R²:   0.9063

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7633 1118.723446  759.469593 0.901113
    ford 13120 1381.416863  908.169055 0.917225
  toyota  3770 1386.423964  860.994271 0.950204
   skoda  3504 1536.559687 1098.264326 0.936526
 hyundai  2728 1552.258830 1006.690762 0.932088
      vw  8488 1978.751757 1327.420570 0.933804
    audi  5979 2916.518209 1908.458404 0.937269
     bmw  6032 3430.589480 2042.193733 0.912205
mercedes  9524 3481.102444 2019.716758 0.901969

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1912 1274.143190  883.549372 0.873773
    ford 3272

,Brand,N,RMSE,MAE,R²
7,opel,1912,1274.143190,883.549372,0.873773
4,ford,3272,1692.398602,1120.063570,0.871391
1,hyundai,676,1751.783234,1207.137574,0.913195
6,toyota,948,2183.386741,1163.818354,0.901834
3,vw,2118,2521.333122,1690.080925,0.898150
8,skoda,882,3076.105669,1479.626531,0.793949
2,audi,1489,3900.317141,2375.428610,0.890086
0,bmw,1514,4080.868240,2518.895905,0.859311
5,mercedes,2384,4508.164805,2600.956711,0.845604


### Random Forest

In [78]:
RF = RandomForestRegressor(random_state=69)
rf_trainer = BrandModelTrainer(RF)


rf_trainer.fit(X_train, y_train)


rf_trainer.evaluate_train(X_train, y_train)
rf_trainer.evaluate(X_val, y_val)

rf_trainer.evaluate_train_by_brand(X_train, y_train)
rf_trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ mercedes done.
  ✓ ford done.
  ✓ opel done.
  ✓ skoda done.
  ✓ audi done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ bmw done.

Training Set Performance (Overall):
  RMSE: 828.36
  MAE:  472.72
  R²:   0.9927

Validation Set Performance (Overall):
  RMSE: 2233.99
  MAE:  1286.08
  R²:   0.9482

Training Performance per Brand:
   Brand     N        RMSE        MAE       R²
    opel  7633  446.549880 295.556575 0.984244
    ford 13120  524.178664 338.974203 0.988082
 hyundai  2728  555.835528 352.943321 0.991292
  toyota  3770  579.052927 347.716891 0.991314
   skoda  3504  588.728238 405.752300 0.990682
      vw  8488  739.931366 460.947176 0.990744
    audi  5979 1027.835403 646.914747 0.992209
mercedes  9524 1181.202633 686.011663 0.988713
     bmw  6032 1221.793276 666.110857 0.988864

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1912 1141.155866  783.097798 0.898748
    ford 3272 1427.626564

,Brand,N,RMSE,MAE,R²
7,opel,1912,1141.155866,783.097798,0.898748
4,ford,3272,1427.626564,919.831189,0.908485
6,toyota,948,1551.581965,891.329684,0.950426
1,hyundai,676,1575.881109,1008.623580,0.929753
3,vw,2118,2014.058237,1282.536838,0.935010
8,skoda,882,2739.298041,1129.830351,0.836600
0,bmw,1514,2785.243281,1750.372695,0.934464
2,audi,1489,2982.593525,1753.850161,0.935725
5,mercedes,2384,3073.058973,1901.756418,0.928257


### Neural Network

In [79]:
from sklearn.neural_network import MLPRegressor

mlp_deep = MLPRegressor(
    hidden_layer_sizes=(256, 128, 64),  
    activation='relu',
    solver='adam',
    alpha=0.0001,  
    learning_rate_init=0.001,
    learning_rate='adaptive',
    max_iter=2000,
    batch_size=256,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=30,  
    validation_fraction=0.15,
    verbose=True
)

mlp_trainer = BrandModelTrainer(mlp_deep)


mlp_trainer.fit(X_train, y_train)


mlp_trainer.evaluate_train(X_train, y_train)
mlp_trainer.evaluate(X_val, y_val)


mlp_trainer.evaluate_train_by_brand(X_train, y_train)
mlp_trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

Iteration 1, loss = 364771944.06513423
Validation score: -5.895844
Iteration 2, loss = 359762314.87924159
Validation score: -5.636126
Iteration 3, loss = 325595247.23774594
Validation score: -4.341845
Iteration 4, loss = 216729778.86876193
Validation score: -1.521373
Iteration 5, loss = 78330282.92392096
Validation score: 0.261328
Iteration 6, loss = 32374601.21564107
Validation score: 0.545695
Iteration 7, loss = 24345913.69186404
Validation score: 0.630332
Iteration 8, loss = 21518155.24057179
Validation score: 0.668546
Iteration 9, loss = 19897585.37890808
Validation score: 0.691826
Iteration 10, loss = 18702125.67444406
Validation score: 0.704042
Iteration 11, loss = 17915964.00727439
Validation score: 0.710169
Iteration 12, loss = 17244805.12727088
Validation score: 0.721765
Iteration 13, loss = 16706856.43732269
Validation score: 0.727891
Iteration 14, loss = 16239235.19005160
Validation score: 0.734724
Iteration 15, loss = 15903477.01230653
Valid

,Brand,N,RMSE,MAE,R²
7,opel,1912,1126.031498,781.354543,0.901414
4,ford,3272,1537.055512,1035.234848,0.893918
1,hyundai,676,1618.566354,1098.226810,0.925896
6,toyota,948,1805.556131,989.849916,0.932869
3,vw,2118,2055.793181,1352.922704,0.932289
8,skoda,882,2791.922977,1221.496772,0.830262
0,bmw,1514,3036.292605,1942.782433,0.922117
2,audi,1489,3334.061488,2004.198121,0.919684
5,mercedes,2384,3518.602942,2299.999134,0.905946


| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3488.30 | 2061.42 | 0.8737 |
| ElasticNet | 4956.03 | 3039.99 | 0.7450 |
| KNR|3004.83 |1691.97 |0.9063 |
|RF | 2233.99 |1286.08 |0.9482 |
|NN | 2445.42 |1442.16 |0.9379 |

 

## Feature selection - Filter Methods

In [113]:
df = X_train.join(y_train)

In [114]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60778 entries, 44735 to 59767
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   mileage           60778 non-null  int32  
 1   tax               60778 non-null  int32  
 2   mpg               60778 non-null  float64
 3   engineSize        60778 non-null  float64
 4   paintQuality%     60778 non-null  int32  
 5   previousOwners    60778 non-null  int32  
 6   stated_no_damage  60778 non-null  float64
 7   Brand             60778 non-null  object 
 8   transmission      60778 non-null  object 
 9   model             60778 non-null  object 
 10  fuelType          60778 non-null  object 
 11  age               60778 non-null  int32  
 12  mileage_per_year  60778 non-null  float64
 13  efficiency_ratio  60778 non-null  float64
 14  age_mileage       60778 non-null  float64
 15  condition_score   60778 non-null  float64
 16  price             60778 non-null  int32  

In [115]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
y = df['price']

numerical_cols = df.select_dtypes(include=['int64', "int32",'float64']).columns

# Drop 'carID' and 'price'
num_cols = [col for col in numerical_cols 
            if "_transformed" not in col and col not in ['price', 'carID']]

print(num_cols)
print(categorical_cols)

['mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'stated_no_damage', 'age', 'mileage_per_year', 'efficiency_ratio', 'age_mileage', 'condition_score']
['Brand', 'transmission', 'model', 'fuelType']


In [116]:
df

,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,stated_no_damage,Brand,transmission,model,fuelType,age,mileage_per_year,efficiency_ratio,age_mileage,condition_score,price
carID,,,,,,,,,,,,,,,,,
44735,1000,145,35.8,2.0,86,2,1.0,mercedes,semi-auto,a class,petrol,4,200.000000,17.047619,0.04000,0.930,33569
44774,5012,116,59.2,1.6,51,3,1.0,mercedes,semi-auto,c class,diesel,5,835.333333,34.823529,0.25060,0.755,27699
36653,10000,146,52.0,1.5,74,3,1.0,mercedes,automatic,c class,petrol,5,1666.666667,32.500000,0.50000,0.870,24400
22889,110000,20,67.3,2.0,61,1,1.0,ford,manual,mondeo,diesel,9,11000.000000,32.047619,9.90000,0.805,6391
62059,17113,145,47.1,1.4,89,4,1.0,opel,manual,mokka x,petrol,6,2444.714286,31.400000,1.02678,0.945,11386
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71271,15773,20,58.9,1.0,90,2,0.0,vw,manual,polo,petrol,7,1971.625000,53.545455,1.10411,0.450,9798
7862,27563,160,46.3,2.0,60,3,1.0,bmw,semi-auto,5 series,petrol,9,2756.300000,22.047619,2.48067,0.800,16750
33501,10164,145,60.1,1.0,40,2,1.0,hyundai,manual,i10,petrol,6,1452.000000,54.636364,0.60984,0.700,7900


In [117]:
#FILTER METHOD


#ANOVA FUNCTION

def anova_for_categorical(df, y, categorical_cols):
    # Align indices between df and y
    common_idx = df.index.intersection(y.index)
    df_aligned = df.loc[common_idx]
    y_aligned = y.loc[common_idx]
    
    f_scores, p_values = [], []
    for col in df_aligned.columns:
        if col in categorical_cols:
            groups = [y_aligned[df_aligned[col] == cat] for cat in df_aligned[col].dropna().unique()]
            if len(groups) > 1 and all(len(g) > 1 for g in groups):
                f_stat, p_val = stats.f_oneway(*groups)
            else:
                f_stat, p_val = 0.0, 1.0
        else:
            if df_aligned[col].nunique() > 1:
                # Remove NaN values for correlation calculation
                valid_idx = df_aligned[col].notna() & y_aligned.notna()
                if valid_idx.sum() > 1:
                    corr = np.corrcoef(df_aligned.loc[valid_idx, col], y_aligned[valid_idx])[0, 1]
                    f_stat = corr**2 * valid_idx.sum()
                    p_val = 0.0
                else:
                    f_stat, p_val = 0.0, 1.0
            else:
                f_stat, p_val = 0.0, 1.0
        f_scores.append(f_stat)
        p_values.append(p_val)
    
    return np.array(f_scores), np.array(p_values)


def filter_method_selection(X_train, y_train, categorical_cols, num_cols,
                            top_k=None,
                            var_threshold=0.01,
                            corr_threshold=0.85):
    
    print("FILTER METHOD (Variance + Spearman Correlation + ANOVA)")
    
    # Ensure indices match
    common_idx = X_train.index.intersection(y_train.index)
    X_train = X_train.loc[common_idx]
    y_train = y_train.loc[common_idx]
    
    # Filter numerical columns that exist in X_train
    num_cols_in_X = [col for col in num_cols if col in X_train.columns]
    
    # Variance Threshold
    if num_cols_in_X:
        vt_selector = VarianceThreshold(threshold=var_threshold)
        X_num_vt = pd.DataFrame(
            vt_selector.fit_transform(X_train[num_cols_in_X]),
            columns=np.array(num_cols_in_X)[vt_selector.get_support()],
            index=X_train.index
        )
        print(f"Removed {len(num_cols_in_X) - X_num_vt.shape[1]} low-variance numeric features.")
    else:
        X_num_vt = pd.DataFrame(index=X_train.index)
    
    # Spearman Correlation
    if not X_num_vt.empty and X_num_vt.shape[1] > 1:
        corr_matrix = X_num_vt.corr(method='spearman').abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [col for col in upper.columns if any(upper[col] > corr_threshold)]
        X_num_corr = X_num_vt.drop(columns=to_drop)
        print(f"Removed {len(to_drop)} correlated numeric features (Spearman |corr| > {corr_threshold}).")
    else:
        X_num_corr = X_num_vt
    
    # Filter categorical columns that exist in X_train
    categorical_cols_in_X = [col for col in categorical_cols if col in X_train.columns]
    
    # Combine numeric + categorical
    X_filtered = pd.concat([X_num_corr, X_train[categorical_cols_in_X]], axis=1)
    
    # ANOVA F-test
    f_scores, f_pvalues = anova_for_categorical(X_filtered, y_train, categorical_cols_in_X)
    f_norm = (f_scores - f_scores.min()) / (f_scores.max() - f_scores.min() + 1e-10)
    
    results_df = pd.DataFrame({
        'feature': X_filtered.columns,
        'ANOVA_F': f_scores,
        'ANOVA_p_value': f_pvalues,
        'ANOVA_norm': f_norm,
        'type': ['categorical' if c in categorical_cols_in_X else 'numerical' for c in X_filtered.columns]
    }).sort_values('ANOVA_norm', ascending=False)
    
    if top_k is None:
        selected_features = X_filtered.columns.tolist()
    else:
        selected_features = X_filtered.columns[np.argsort(f_norm)[-top_k:]].tolist()
    
    print(f"\nFilter method selected {len(selected_features)} features")
    
    return selected_features, X_filtered[selected_features], results_df



In [118]:
y = df["price"]

df = df.drop(columns=["price"])

In [119]:

#X_train = df[num_cols]
#y_train = df['price']

# Filter method (Variance + Spearman + ANOVA)
# Define categorical and numerical columns first (if not already defined)
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = df.select_dtypes(include=['int64',"int32", 'float64']).columns.tolist()

# Filter method with all required parameters
selected_filter, X_train_filtered, filter_df = filter_method_selection(
    df, 
    y,
    categorical_cols=categorical_cols,  # Add this
    num_cols=num_cols,                  # Add this
    top_k=None, 
    var_threshold=0.01, 
    corr_threshold=0.85
)

# Print features selected
print("\nFeatures selected by Filter Method:")
for f in selected_filter:
    print(f)

# Display feature importance scores
print("\nTop 10 Features by ANOVA Score:")
print(filter_df.head(10))



FILTER METHOD (Variance + Spearman Correlation + ANOVA)
Removed 0 low-variance numeric features.
Removed 3 correlated numeric features (Spearman |corr| > 0.85).

Filter method selected 13 features

Features selected by Filter Method:
mileage
tax
mpg
engineSize
paintQuality%
previousOwners
stated_no_damage
age
efficiency_ratio
Brand
transmission
model
fuelType

Top 10 Features by ANOVA Score:
             feature       ANOVA_F  ANOVA_p_value  ANOVA_norm         type
3         engineSize  23128.081112            0.0    1.000000    numerical
7                age  13914.885189            0.0    0.601645    numerical
10      transmission  12694.800173            0.0    0.548891  categorical
0            mileage  10456.747359            0.0    0.452123    numerical
1                tax   6204.904680            0.0    0.268284    numerical
2                mpg   5267.455866            0.0    0.227752    numerical
9              Brand   3113.772851            0.0    0.134632  categorical
12   

In [120]:
raise SystemExit("Stop before training the models")

SystemExit: Stop before training the models

C:\Users\liber\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Test models on reduced DF

In [88]:
X_train_filter = X_train[["mileage", "tax", "mpg", "engineSize", "paintQuality%", "previousOwners", "stated_no_damage", "age", "efficiency_ratio",
                            "Brand", "transmission", "model", "fuelType"]]
X_train_filter

,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,stated_no_damage,age,efficiency_ratio,Brand,transmission,model,fuelType
carID,,,,,,,,,,,,,
44735,1000,145,35.8,2.0,86,2,1.0,4,17.047619,mercedes,semi-auto,a class,petrol
44774,5012,116,59.2,1.6,51,3,1.0,5,34.823529,mercedes,semi-auto,c class,diesel
36653,10000,146,52.0,1.5,74,3,1.0,5,32.500000,mercedes,automatic,c class,petrol
22889,110000,20,67.3,2.0,61,1,1.0,9,32.047619,ford,manual,mondeo,diesel
62059,17113,145,47.1,1.4,89,4,1.0,6,31.400000,opel,manual,mokka x,petrol
...,...,...,...,...,...,...,...,...,...,...,...,...,...
71271,15773,20,58.9,1.0,90,2,0.0,7,53.545455,vw,manual,polo,petrol
7862,27563,160,46.3,2.0,60,3,1.0,9,22.047619,bmw,semi-auto,5 series,petrol
33501,10164,145,60.1,1.0,40,2,1.0,6,54.636364,hyundai,manual,i10,petrol


In [89]:
X_val_filter = X_val[["mileage", "tax", "mpg", "engineSize", "paintQuality%", "previousOwners", "stated_no_damage", "age", "efficiency_ratio",
                            "Brand", "transmission", "model", "fuelType"]]
X_val_filter

,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,stated_no_damage,age,efficiency_ratio,Brand,transmission,model,fuelType
carID,,,,,,,,,,,,,
10742,10,145,60.1,2.0,46,2,1.0,5,28.619048,bmw,manual,2 series,diesel
33830,26671,145,44.8,1.6,40,1,1.0,7,26.352941,hyundai,manual,tucson,petrol
2417,148000,305,35.8,3.0,82,0,1.0,16,11.548387,audi,manual,a4,diesel
4544,6541,145,47.9,2.0,59,2,1.0,5,22.809524,audi,semi-auto,a4,diesel
73794,31190,145,58.9,1.0,73,0,1.0,6,53.545455,vw,manual,golf sv,petrol
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8206,32285,150,68.9,2.0,98,4,1.0,7,32.809524,bmw,manual,1 series,diesel
71796,3252,145,32.8,2.0,71,3,1.0,5,15.619048,vw,semi-auto,golf,petrol
72896,67039,145,48.7,1.8,40,2,1.0,15,25.631579,vw,manual,polo,petrol


In [90]:
LR = LinearRegression()
trainer = BrandModelTrainer(LR)

# fit
trainer.fit(X_train_filter, y_train)

# performance overall
trainer.evaluate_train(X_train_filter, y_train)
trainer.evaluate(X_val_filter, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train_filter, y_train)
trainer.evaluate_by_brand(X_val_filter, y_val)

Training models for 9 brands...

  ✓ mercedes done.
  ✓ ford done.
  ✓ opel done.
  ✓ skoda done.
  ✓ audi done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ bmw done.

Training Set Performance (Overall):
  RMSE: 3505.92
  MAE:  2119.84
  R²:   0.8698

Validation Set Performance (Overall):
  RMSE: 3611.32
  MAE:  2167.81
  R²:   0.8646

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7633 1587.460508 1099.919342 0.800886
  toyota  3770 1922.071099 1183.028084 0.904293
    ford 13120 2122.843582 1512.673343 0.804527
   skoda  3504 2134.574421 1516.083684 0.877506
 hyundai  2728 2197.747433 1538.439900 0.863864
      vw  8488 2916.108755 2037.931265 0.856233
    audi  5979 4338.783014 2817.498954 0.861169
     bmw  6032 4530.372664 3018.630939 0.846892
mercedes  9524 5770.013105 3598.940739 0.730672

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1912 1601.653529 1125.376930 0.800542
    ford 3272

,Brand,N,RMSE,MAE,R²
7,opel,1912,1601.653529,1125.376930,0.800542
4,ford,3272,2072.927484,1503.594797,0.807055
1,hyundai,676,2290.049307,1582.255137,0.851655
6,toyota,948,2334.180053,1263.668663,0.887806
3,vw,2118,3016.630278,2114.531143,0.854204
8,skoda,882,3127.743839,1604.868756,0.786973
2,audi,1489,4301.443655,2780.241433,0.866315
0,bmw,1514,4330.239245,2984.125322,0.841591
5,mercedes,2384,6028.838676,3795.743667,0.723877


In [91]:
Elastic = ElasticNet()
trainer = BrandModelTrainer(Elastic)

# fit
trainer.fit(X_train_filter, y_train)

# performance overall
trainer.evaluate_train(X_train_filter, y_train)
trainer.evaluate(X_val_filter, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train_filter, y_train)
trainer.evaluate_by_brand(X_val_filter, y_val)        # Performance VALID

Training models for 9 brands...

  ✓ mercedes done.
  ✓ ford done.
  ✓ opel done.
  ✓ skoda done.
  ✓ audi done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ bmw done.

Training Set Performance (Overall):
  RMSE: 4942.71
  MAE:  3033.19
  R²:   0.7413

Validation Set Performance (Overall):
  RMSE: 5009.89
  MAE:  3070.02
  R²:   0.7395

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7633 2307.406147 1726.696222 0.579329
    ford 13120 2790.398116 2006.953052 0.662260
   skoda  3504 3427.075818 2452.132851 0.684252
 hyundai  2728 3444.723415 2475.356616 0.665553
  toyota  3770 3588.944737 2384.883692 0.666315
      vw  8488 4495.913785 3090.974731 0.658267
    audi  5979 6456.733053 3834.177629 0.692550
mercedes  9524 6832.184248 4353.053290 0.622386
     bmw  6032 7666.728779 4954.377269 0.561519

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1912 2280.690821 1729.146369 0.595566
    ford 3272

,Brand,N,RMSE,MAE,R²
7,opel,1912,2280.690821,1729.146369,0.595566
4,ford,3272,2693.744091,1993.583232,0.674181
1,hyundai,676,3424.031652,2401.232847,0.668366
8,skoda,882,4229.256080,2659.747942,0.610507
6,toyota,948,4367.256157,2547.810722,0.607248
3,vw,2118,4671.112170,3249.607287,0.650423
2,audi,1489,6371.030489,3801.402761,0.706726
0,bmw,1514,7072.751441,4724.489407,0.577397
5,mercedes,2384,7198.090578,4504.861671,0.606387


In [92]:
KNR = KNeighborsRegressor()
trainer = BrandModelTrainer(KNR)

# fit
trainer.fit(X_train_filter, y_train)

# performance overall
trainer.evaluate_train(X_train_filter, y_train)
trainer.evaluate(X_val_filter, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train_filter, y_train)
trainer.evaluate_by_brand(X_val_filter, y_val)

Training models for 9 brands...

  ✓ mercedes done.
  ✓ ford done.
  ✓ opel done.
  ✓ skoda done.
  ✓ audi done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ bmw done.

Training Set Performance (Overall):
  RMSE: 2331.86
  MAE:  1341.70
  R²:   0.9424

Validation Set Performance (Overall):
  RMSE: 2990.28
  MAE:  1691.12
  R²:   0.9072

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7633 1110.463353  748.398533 0.902568
  toyota  3770 1332.202473  832.235968 0.954023
    ford 13120 1356.013314  883.019497 0.920241
   skoda  3504 1494.834582 1071.983048 0.939927
 hyundai  2728 1526.767346 1001.357405 0.934300
      vw  8488 1928.038340 1318.974741 0.937153
    audi  5979 2979.969927 1935.642014 0.934510
     bmw  6032 3438.592051 2052.168170 0.911795
mercedes  9524 3551.069890 2044.892566 0.897989

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1912 1313.098665  909.123640 0.865937
    ford 3272

,Brand,N,RMSE,MAE,R²
7,opel,1912,1313.098665,909.123640,0.865937
4,ford,3272,1633.083094,1082.981357,0.880249
1,hyundai,676,1865.842886,1275.730769,0.901523
6,toyota,948,1938.332689,1095.546624,0.922633
3,vw,2118,2447.721156,1661.948253,0.904010
8,skoda,882,3010.026979,1401.473016,0.802706
2,audi,1489,3950.739859,2409.848892,0.887225
0,bmw,1514,4118.586179,2566.181242,0.856698
5,mercedes,2384,4502.466256,2636.039681,0.845995


In [93]:
RF = RandomForestRegressor(random_state=69)
rf_trainer = BrandModelTrainer(RF)


rf_trainer.fit(X_train_filter, y_train)


rf_trainer.evaluate_train(X_train_filter, y_train)
rf_trainer.evaluate(X_val_filter, y_val)

rf_trainer.evaluate_train_by_brand(X_train_filter, y_train)
rf_trainer.evaluate_by_brand(X_val_filter, y_val)

Training models for 9 brands...

  ✓ mercedes done.
  ✓ ford done.
  ✓ opel done.
  ✓ skoda done.
  ✓ audi done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ bmw done.

Training Set Performance (Overall):
  RMSE: 825.28
  MAE:  471.41
  R²:   0.9928

Validation Set Performance (Overall):
  RMSE: 2199.79
  MAE:  1275.63
  R²:   0.9498

Training Performance per Brand:
   Brand     N        RMSE        MAE       R²
    opel  7633  446.553515 296.280294 0.984244
    ford 13120  521.334117 337.928879 0.988211
 hyundai  2728  556.679215 354.290044 0.991266
  toyota  3770  580.918446 347.435915 0.991258
   skoda  3504  599.534590 407.890890 0.990337
      vw  8488  729.062862 459.621713 0.991014
    audi  5979 1024.328063 643.965190 0.992262
mercedes  9524 1179.741378 683.904246 0.988741
     bmw  6032 1213.975644 660.730448 0.989006

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1912 1138.584807  782.090724 0.899204
    ford 3272 1409.114654

,Brand,N,RMSE,MAE,R²
7,opel,1912,1138.584807,782.090724,0.899204
4,ford,3272,1409.114654,914.551974,0.910843
1,hyundai,676,1564.568370,1012.588565,0.930758
6,toyota,948,1578.062076,907.960496,0.948720
3,vw,2118,1973.198928,1265.436747,0.937620
8,skoda,882,2739.936003,1116.640091,0.836524
0,bmw,1514,2786.715586,1744.709122,0.934394
2,audi,1489,2927.764329,1748.242962,0.938066
5,mercedes,2384,2978.509295,1862.620331,0.932604


In [94]:
from sklearn.neural_network import MLPRegressor

mlp_deep = MLPRegressor(
    hidden_layer_sizes=(256, 128, 64),  
    activation='relu',
    solver='adam',
    alpha=0.0001,  
    learning_rate_init=0.001,
    learning_rate='adaptive',
    max_iter=2000,
    batch_size=256,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=30,  
    validation_fraction=0.15,
    verbose=True
)

mlp_trainer = BrandModelTrainer(mlp_deep)


mlp_trainer.fit(X_train_filter, y_train)


mlp_trainer.evaluate_train(X_train_filter, y_train)
mlp_trainer.evaluate(X_val_filter, y_val)


mlp_trainer.evaluate_train_by_brand(X_train_filter, y_train)
mlp_trainer.evaluate_by_brand(X_val_filter, y_val)

Training models for 9 brands...

Iteration 1, loss = 363484396.04617161
Validation score: -5.687736
Iteration 2, loss = 359248626.43689102
Validation score: -5.477660
Iteration 3, loss = 332130181.63105667
Validation score: -4.460717
Iteration 4, loss = 242595327.75663561
Validation score: -2.046251
Iteration 5, loss = 107113782.12193915
Validation score: -0.028550
Iteration 6, loss = 42734994.25602884
Validation score: 0.384009
Iteration 7, loss = 28949559.18862030
Validation score: 0.547595
Iteration 8, loss = 23465120.67177410
Validation score: 0.616564
Iteration 9, loss = 20964133.03023534
Validation score: 0.652100
Iteration 10, loss = 19534923.15288718
Validation score: 0.680525
Iteration 11, loss = 18629790.00127812
Validation score: 0.695709
Iteration 12, loss = 18034208.93990591
Validation score: 0.704869
Iteration 13, loss = 17473906.81268818
Validation score: 0.716191
Iteration 14, loss = 17052548.27323876
Validation score: 0.723869
Iteration 15, loss = 16697131.00582343
Val

,Brand,N,RMSE,MAE,R²
7,opel,1912,1186.500705,800.923160,0.890541
4,ford,3272,1539.916948,1031.839193,0.893522
1,hyundai,676,1655.840881,1103.776241,0.922443
6,toyota,948,1882.418828,1031.582812,0.927032
3,vw,2118,2093.184574,1387.211827,0.929803
8,skoda,882,2803.950805,1219.084165,0.828796
0,bmw,1514,3168.423637,2047.445679,0.915191
2,audi,1489,3337.461660,2003.108940,0.919520
5,mercedes,2384,3541.420207,2315.413045,0.904723


Performance on the full dataset

| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3488.30 | 2061.42 | 0.8737 |
| ElasticNet | 4956.03 | 3039.99 | 0.7450 |
| KNR|3004.83 |1691.97 |0.9063 |
|RF | 2233.99 |1286.08 |0.9482 |
|NN | 2445.42 |1442.16 |0.9379 |

Performance on the reduced dataset
| *Model* | *RMSE* | *MAE* | *R²* |
|---------|--------|--------|------|
| Linear Regression | 3611.32 | 2167.81 | 0.8646 |
| ElasticNet        | 5009.89 | 3070.02 | 0.7395 |
| KNR               | 2990.28 | 1691.12 | 0.9072 |
| RF                | **2199.79** | **1275.63** | **0.9498** |
| NN                | 2481.41 | 1464.13 | 0.9361 |




In [95]:
raise SystemExit("Stop before training the models")

SystemExit: Stop before training the models

C:\Users\liber\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Feature Selection - Wrapper Method

In [ ]:
#WRAPPER METHOD

def rfe(train_processed, validation_processed, num_cols, step=1, n_estimators=100, random_state=42):
    
    # Filter numeric columns to those that actually exist in train_processed
    valid_num_cols = [col for col in num_cols if col in train_processed.columns]
    if len(valid_num_cols) == 0:
        raise ValueError("No valid numeric columns found in train_processed.")

    print(f"Using {len(valid_num_cols)} numeric columns for RFE:\n{valid_num_cols}")

    # Prepare numeric features and targets
    X_train_num = train_processed[valid_num_cols]
    y_train = train_processed['price']

    X_val_num = validation_processed[valid_num_cols]
    y_val = validation_processed['price']

    nof_list = np.arange(1, X_train_num.shape[1]+1)
    high_score = 0
    nof = 0
    train_score_list = []
    val_score_list = []
    features_to_select = None

    for n in nof_list:
        model = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state, n_jobs=-1)
        rfe = RFE(estimator=model, n_features_to_select=n, step=step)
        X_train_rfe = rfe.fit_transform(X_train_num, y_train)
        X_val_rfe = rfe.transform(X_val_num)
        model.fit(X_train_rfe, y_train)

        train_score = model.score(X_train_rfe, y_train)
        val_score = model.score(X_val_rfe, y_val)
        train_score_list.append(train_score)
        val_score_list.append(val_score)

        if val_score > high_score:
            high_score = val_score
            nof = n
            features_to_select = pd.Series(rfe.support_, index=X_train_num.columns)

    selected_features = features_to_select[features_to_select].index.tolist()
    
    print
    print(f"\nOptimum number of features: {nof}")
    print(f"Best validation score: {high_score:.4f}")
    print("Selected features:")
    print(selected_features)

    return nof, high_score, selected_features, train_score_list, val_score_list


In [ ]:

# RFE (Random Forest) using your workflow variables
nof, best_score, selected_rfe_features, train_scores, val_scores = rfe(
    train_processed, validation_processed, num_cols
)

Using 7 numeric columns for RFE:
['mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'stated_no_damage']

Optimum number of features: 5
Best validation score: 0.8478
Selected features:
['mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%']


## Hyperparam tuning

In [99]:
class HoldoutRandomSearch:
    def __init__(self, trainer_class, param_space, n_iter=20, optimize_metric='mae'):
        self.trainer_class = trainer_class
        self.param_space = param_space
        self.n_iter = n_iter
        self.optimize_metric = optimize_metric.lower()
        self.results = []
        self.best_params = None
        self.best_score = None
        self.best_trainer = None
        self.brand_best_configs = {}
        
    def sample_params(self):
        if isinstance(self.param_space, list):
            return random.choice(self.param_space)
        else:
            return {k: random.choice(v) for k, v in self.param_space.items()}
    
    def _get_param_summary(self, params):
        summary = {}
        
        if 'n_estimators' in params:
            summary['n_estimators'] = params['n_estimators']
        if 'max_depth' in params:
            summary['max_depth'] = params['max_depth']
        if 'max_features' in params:
            summary['max_features'] = params['max_features']
            
        if 'hidden_layer_sizes' in params:
            summary['hidden_layers'] = str(params['hidden_layer_sizes'])
        if 'alpha' in params:
            summary['alpha'] = params['alpha']
        if 'learning_rate_init' in params:
            summary['lr'] = params['learning_rate_init']
        if 'activation' in params:
            summary['activation'] = params['activation']
            
        return summary
    
    def run(self, X_train, y_train, X_val, y_val):
        metric_name = self.optimize_metric.upper()
        print(f"Running random search ({self.n_iter} iterations)...")
        print(f"Optimizing for: {metric_name}\n")
        
        start_time = time.time()
        brands = X_train["Brand"].unique()

        for brand in brands:
            self.brand_best_configs[brand] = {
                'score': float('inf'),
                'params': None,
                'config_num': None,
                'all_metrics': {}
            }
        
        for i in range(self.n_iter):
            iter_start = time.time()
            params = self.sample_params()
            
            print(f"\n{'='*70}")
            print(f"[{i+1}/{self.n_iter}] Testing params:")
            print(params)
            print('='*70)
            

            estimator = self.trainer_class.estimator.__class__(**params)
            trainer = BrandModelTrainer(estimator)
            trainer.fit(X_train, y_train)

            metrics = trainer.evaluate(X_val, y_val)
            rmse = metrics["RMSE"]
            mae = metrics["MAE"]
            r2 = metrics["R²"]
            
            current_score = mae if self.optimize_metric == 'mae' else rmse
            
            brand_results = {}
            for brand in brands:
                mask = X_val["Brand"] == brand
                y_true_brand = y_val[mask]
                y_pred_brand = trainer.predict(X_val[mask])
                
                brand_rmse = np.sqrt(mean_squared_error(y_true_brand, y_pred_brand))
                brand_mae = mean_absolute_error(y_true_brand, y_pred_brand)
                brand_r2 = r2_score(y_true_brand, y_pred_brand)
                
                brand_score = brand_mae if self.optimize_metric == 'mae' else brand_rmse
                
                brand_results[brand] = {
                    'score': brand_score,
                    'rmse': brand_rmse,
                    'mae': brand_mae,
                    'r2': brand_r2
                }
                
                if brand_score < self.brand_best_configs[brand]['score']:
                    self.brand_best_configs[brand]['score'] = brand_score
                    self.brand_best_configs[brand]['params'] = params.copy()
                    self.brand_best_configs[brand]['config_num'] = i + 1
                    self.brand_best_configs[brand]['all_metrics'] = {
                        'rmse': brand_rmse,
                        'mae': brand_mae,
                        'r2': brand_r2
                    }
            
            self.results.append({
                "config_num": i + 1,
                "params": params,
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
                "score": current_score,
                "brand_results": brand_results
            })

            if self.best_score is None or current_score < self.best_score:
                self.best_score = current_score
                self.best_params = params
                self.best_trainer = trainer
                print(f"New best overall model ({metric_name}: {current_score:.2f})")
           

            elapsed_total = time.time() - start_time
            avg_per_iter = elapsed_total / (i + 1)
            eta = avg_per_iter * (self.n_iter - (i + 1))
            
            print(f"\nProgress: {i+1}/{self.n_iter} | Elapsed: {elapsed_total:.1f}s | ETA: ~{eta:.1f}s")
        
        print(f"SEARCH COMPLETED - FINAL RESULTS (Optimized for {metric_name})")
        
        print(f"\n Best general model:")
        print(f"  Best {metric_name}: {self.best_score:.2f}")
        
        best_result = [r for r in self.results if r['score'] == self.best_score][0]
        print(f"  RMSE: {best_result['rmse']:.2f}")
        print(f"  MAE:  {best_result['mae']:.2f}")
        print(f"  R²:   {best_result['r2']:.4f}")
        print(f"  Params: {self.best_params}")
   
        print(f"Best configuration per brand (by {metric_name})")

        
        brand_summary = []
        for brand in sorted(brands):
            config = self.brand_best_configs[brand]
            
            summary = {
                'Brand': brand,
                f'Best_{metric_name}': config['score'],
                'RMSE': config['all_metrics']['rmse'],
                'MAE': config['all_metrics']['mae'],
                'R²': config['all_metrics']['r2'],
                'Config_Num': config['config_num']
            }
            
            param_summary = self._get_param_summary(config['params'])
            summary.update(param_summary)
            
            brand_summary.append(summary)
            
            print(f"\n{brand.upper()}:")
            print(f"  Best {metric_name}: {config['score']:.2f}")
            print(f"  RMSE: {config['all_metrics']['rmse']:.2f}")
            print(f"  MAE:  {config['all_metrics']['mae']:.2f}")
            print(f"  R²:   {config['all_metrics']['r2']:.4f}")
            print(f"  Found at iteration: {config['config_num']}")
            print(f"  Best params:")
            for k, v in list(config['params'].items())[:5]:
                if k not in ['random_state', 'n_jobs', 'shuffle', 'verbose', 'warm_start']:
                    print(f"    {k}: {v}")
        
        df_summary = pd.DataFrame(brand_summary).sort_values(f'Best_{metric_name}')
  
        print(df_summary.to_string(index=False))
        
        results_df = pd.DataFrame([
            {
                'Config': r['config_num'],
                metric_name: r['score'],
                'RMSE': r['rmse'],
                'MAE': r['mae'],
                'R²': r['r2']
            }
            for r in self.results
        ]).sort_values(metric_name)
        
        print(f"ALL CONFIGURATIONS (sorted by {metric_name}):")
        print(results_df.head(10).to_string(index=False))
        
        return self.best_trainer, self.best_params, self.best_score

### Random Forest

In [103]:
base_estimator = RandomForestRegressor(random_state=69, n_jobs=-1)
trainer = BrandModelTrainer(base_estimator)

param_space_rf = {
    "n_estimators": [400, 600, 1000, 1200],
    "max_depth": [None, 10, 20, 40, 50, 60],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [0.6, 0.8, 0.9],
    "min_impurity_decrease": [0.0, 0.5, 1, 5],
    "max_samples": [0.85, 0.9, None],
    "ccp_alpha": [0.0], 
    "random_state": [69],
    "n_jobs": [-1]
}

search = HoldoutRandomSearch(
    trainer_class=trainer,
    param_space=param_space_rf,
    n_iter=10
)


best_trainer, best_params, best_rmse = search.run(X_train_filter, y_train, X_val_filter, y_val)


Running random search (10 iterations)...
Optimizing for: MAE


[1/10] Testing params:
{'n_estimators': 1200, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.9, 'min_impurity_decrease': 0.0, 'max_samples': 0.9, 'ccp_alpha': 0.0, 'random_state': 69, 'n_jobs': -1}
Training models for 9 brands...

  ✓ mercedes done.
  ✓ ford done.
  ✓ opel done.
  ✓ skoda done.
  ✓ audi done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ bmw done.

Validation Set Performance (Overall):
  RMSE: 2304.25
  MAE:  1374.08
  R²:   0.9449
New best overall model (MAE: 1374.08)

Progress: 1/10 | Elapsed: 151.0s | ETA: ~1358.9s

[2/10] Testing params:
{'n_estimators': 1200, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.9, 'min_impurity_decrease': 5, 'max_samples': 0.9, 'ccp_alpha': 0.0, 'random_state': 69, 'n_jobs': -1}
Training models for 9 brands...

  ✓ mercedes done.
  ✓ ford done.
  ✓ opel done.
  ✓ skoda done.
  ✓ audi done.
  ✓ toyo

In [104]:
best_trainer.evaluate_by_brand(X_val, y_val)



Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1912 1111.904671  766.004723 0.903872
    ford 3272 1386.288222  903.570179 0.913708
 hyundai  676 1514.489790  987.232646 0.935119
  toyota  948 1568.562553  899.105577 0.949335
      vw 2118 1947.857993 1242.387070 0.939212
   skoda  882 2719.428060 1094.002885 0.838962
     bmw 1514 2759.002995 1722.280598 0.935693
    audi 1489 2889.571404 1735.413536 0.939672
mercedes 2384 2942.959370 1847.519375 0.934203


,Brand,N,RMSE,MAE,R²
7,opel,1912,1111.904671,766.004723,0.903872
4,ford,3272,1386.288222,903.570179,0.913708
1,hyundai,676,1514.489790,987.232646,0.935119
6,toyota,948,1568.562553,899.105577,0.949335
3,vw,2118,1947.857993,1242.387070,0.939212
8,skoda,882,2719.428060,1094.002885,0.838962
0,bmw,1514,2759.002995,1722.280598,0.935693
2,audi,1489,2889.571404,1735.413536,0.939672
5,mercedes,2384,2942.959370,1847.519375,0.934203


In [ ]:
configs = [

    {'n_estimators': 500, 'max_depth': 40, 'min_samples_split': 2, 
     'min_samples_leaf': 1, 'max_features': 0.85, 'random_state': 69, 'n_jobs': -1},
    
  
    {'n_estimators': 800, 'max_depth': 40, 'min_samples_split': 2, 
     'min_samples_leaf': 1, 'max_features': 0.8, 'random_state': 69, 'n_jobs': -1},
    
    
    {'n_estimators': 600, 'max_depth': 50, 'min_samples_split': 2, 
     'min_samples_leaf': 1, 'max_features': 0.8, 'random_state': 69, 'n_jobs': -1},
    
  
    {'n_estimators': 400, 'max_depth': 45, 'min_samples_split': 2, 
     'min_samples_leaf': 1, 'max_features': 0.9, 'random_state': 69, 'n_jobs': -1},
    

    {'n_estimators': 1000, 'max_depth': 35, 'min_samples_split': 3, 
     'min_samples_leaf': 2, 'max_features': 0.8, 'random_state': 69, 'n_jobs': -1},
    
 
    {'n_estimators': 700, 'max_depth': 40, 'min_samples_split': 3, 
     'min_samples_leaf': 1, 'max_features': 0.85, 'random_state': 69, 'n_jobs': -1},
    

    {'n_estimators': 500, 'max_depth': 60, 'min_samples_split': 2, 
     'min_samples_leaf': 1, 'max_features': 0.75, 'random_state': 69, 'n_jobs': -1},
    

    {'n_estimators': 600, 'max_depth': 40, 'min_samples_split': 5, 
     'min_samples_leaf': 1, 'max_features': 0.7, 'random_state': 69, 'n_jobs': -1},
    

    {'n_estimators': 1200, 'max_depth': 40, 'min_samples_split': 2, 
     'min_samples_leaf': 1, 'max_features': 0.8, 'random_state': 69, 'n_jobs': -1},
    

    {'n_estimators': 600, 'max_depth': None, 'min_samples_split': 2, 
     'min_samples_leaf': 1, 'max_features': 0.8, 'random_state': 69, 'n_jobs': -1},

    {
        #'criterion': 'absolute_error',
        'n_estimators': 600,
        'max_depth': 40,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.7,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'ccp_alpha': 0.0,
        'random_state': 69,
        'n_jobs': -1
    },
    
    {
        #'criterion': 'absolute_error',
        'n_estimators': 800,
        'max_depth': 40,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.7,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'ccp_alpha': 0.0,
        'random_state': 69,
        'n_jobs': -1
    },
    
    {
       # 'criterion': 'absolute_error',
        'n_estimators': 600,
        'max_depth': 40,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.7,
        'min_impurity_decrease': 0.0,
        'max_samples': 0.85,  
        'ccp_alpha': 0.0,
        'random_state': 69,
        'n_jobs': -1
    },
    
    {
        #'criterion': 'absolute_error',
        'n_estimators': 600,
        'max_depth': 40,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.7,
        'min_impurity_decrease': 1.0,
        'max_samples': None,
        'ccp_alpha': 0.0,
        'random_state': 69,
        'n_jobs': -1
    },
    
    {
        #'criterion': 'absolute_error',
        'n_estimators': 600,
        'max_depth': 40,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.8,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'ccp_alpha': 0.0,
        'random_state': 69,
        'n_jobs': -1
    },
    
    {
       # 'criterion': 'absolute_error',
        'n_estimators': 600,
        'max_depth': 50,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.7,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'ccp_alpha': 0.0,
        'random_state': 69,
        'n_jobs': -1
    },
    
    {
        #'criterion': 'absolute_error',
        'n_estimators': 700,
        'max_depth': 40,
        'min_samples_split': 5,
        'min_samples_leaf': 1,
        'max_features': 0.75,
        'min_impurity_decrease': 0.5,
        'max_samples': 0.9,
        'ccp_alpha': 0.0,
        'random_state': 69,
        'n_jobs': -1
    },
    

    {
        #'criterion': 'absolute_error',
        'n_estimators': 600,
        'max_depth': 40,
        'min_samples_split': 5,
        'min_samples_leaf': 2,
        'max_features': 0.7,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'ccp_alpha': 0.0,
        'random_state': 69,
        'n_jobs': -1
    },

    {
        #'criterion': 'absolute_error',
        'n_estimators': 600,
        'max_depth': 45,
        'min_samples_split': 3,
        'min_samples_leaf': 1,
        'max_features': 0.7,
        'min_impurity_decrease': 5.0,
        'max_samples': None,
        'ccp_alpha': 0.0,
        'random_state': 69,
        'n_jobs': -1
    },
    
    {
        #'criterion': 'absolute_error',
        'n_estimators': 1000,
        'max_depth': 45,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'max_features': 0.8,
        'min_impurity_decrease': 0.0,
        'max_samples': None,
        'ccp_alpha': 0.0,
        'random_state': 69,
        'n_jobs': -1
    }
]


best_rmse = float('inf')
best_params = None
best_trainer = None
all_results = []


for i, params in enumerate(configs, 1):
    print(f"\n{'='*70}")
    print(f"[{i}/10] Testing Configuration {i}")
    print(f"{'='*70}")
    print(f"Parameters:")
    for k, v in params.items():
        if k not in ['random_state', 'n_jobs']:
            print(f"  {k}: {v}")
    

    rf = RandomForestRegressor(**params)
    trainer = BrandModelTrainer(estimator=rf)
    

    trainer.fit(X_train_filter, y_train)
    

    val_metrics = trainer.evaluate(X_val_filter, y_val)

    result = {
        'config_num': i,
        'params': params.copy(),
        'rmse': val_metrics['RMSE'],
        'mae': val_metrics['MAE'],
        'r2': val_metrics['R²']
    }
    all_results.append(result)
    

    if val_metrics['RMSE'] < best_rmse:
        best_rmse = val_metrics['RMSE']
        best_params = params
        best_trainer = trainer
        print(" NEW BEST MODEL")



results_df = pd.DataFrame([
    {
        'Config': r['config_num'],
        'RMSE': r['rmse'],
        'MAE': r['mae'],
        'R²': r['r2'],
        'n_estimators': r['params']['n_estimators'],
        'max_depth': r['params']['max_depth'],
        'max_features': r['params']['max_features']
    }
    for r in all_results
]).sort_values('RMSE')

print("\nAll Results (sorted by RMSE):")
print(results_df.to_string(index=False))


print(f"Configuration: #{results_df.iloc[0]['Config']}")
print(f"RMSE: {best_rmse:.2f}")
print(f"\nBest Parameters:")
for k, v in best_params.items():
    if k not in ['random_state', 'n_jobs']:
        print(f"  {k}: {v}")


print(f"\n{'='*70}")
print("BEST MODEL - DETAILED EVALUATION")
print(f"{'='*70}")
best_trainer.evaluate_by_brand(X_val_filter, y_val, split_name="Validation")


[1/10] Testing Configuration 1
Parameters:
  n_estimators: 500
  max_depth: 40
  min_samples_split: 2
  min_samples_leaf: 1
  max_features: 0.85
Training models for 9 brands...

  ✓ mercedes done.
  ✓ ford done.
  ✓ opel done.
  ✓ skoda done.
  ✓ audi done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ bmw done.

Validation Set Performance (Overall):
  RMSE: 2189.30
  MAE:  1272.13
  R²:   0.9502
 NEW BEST MODEL

[2/10] Testing Configuration 2
Parameters:
  n_estimators: 800
  max_depth: 40
  min_samples_split: 2
  min_samples_leaf: 1
  max_features: 0.8
Training models for 9 brands...

  ✓ mercedes done.
  ✓ ford done.
  ✓ opel done.
  ✓ skoda done.
  ✓ audi done.
  ✓ toyota done.
  ✓ hyundai done.
  ✓ vw done.
  ✓ bmw done.

Validation Set Performance (Overall):
  RMSE: 2181.40
  MAE:  1268.27
  R²:   0.9506
 NEW BEST MODEL

[3/10] Testing Configuration 3
Parameters:
  n_estimators: 600
  max_depth: 50
  min_samples_split: 2
  min_samples_leaf: 1
  max_features: 0.8
Training m

,Brand,N,RMSE,MAE,R²
7,opel,1912,1100.085079,759.352091,0.905905
4,ford,3272,1382.574221,901.469981,0.914170
1,hyundai,676,1465.116593,968.086540,0.939281
6,toyota,948,1606.752309,912.695746,0.946838
3,vw,2118,1930.362238,1238.666120,0.940299
8,skoda,882,2712.636379,1085.090500,0.839766
0,bmw,1514,2721.503842,1716.882855,0.937429
2,audi,1489,2931.014349,1772.308270,0.937929
5,mercedes,2384,2970.204540,1862.687096,0.932979


In [ ]:
raise SystemExit("Stop before training the models")

## Predictions

In [ ]:
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

trainer = BrandModelTrainer(estimator=rf)

trainer.fit(X_train, y_train)
trainer.evaluate(X_val, y_val)
trainer.evaluate_by_brand(X_val, y_val)

trainer.save_predictions(X_val, "predictions.csv")